In [ ]:
%pip install -q \
    "datasets==3.6.0" \
    "transformers==4.51.3" \
    "trl==0.17.0" \
    "peft==0.15.2" \
    "accelerate==1.6.0" \
    "bitsandbytes>=0.46.1" \
    sentencepiece huggingface_hub sacrebleu rouge-score bert-score sentence-transformers nltk textstat safetensors


In [ ]:
import os
import gc
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import nltk
import sacrebleu
import textstat

from google.colab import drive, userdata
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    set_seed
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU runtime before running this notebook.")
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")


In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Add your Hugging Face access token to Colab Secrets as HF_TOKEN and enable notebook access.")

BASE_DIR = Path("/content/drive/MyDrive/Clinical_NLP_Project")
RESULTS_DIR = BASE_DIR / "results" / "2K"
CHECKPOINT_DIR = RESULTS_DIR / "training_checkpoints"
ADAPTER_DIR = BASE_DIR / "llama32_1b_radiology_2k_lora_adapter"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
DATASET_NAME = "BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track"
SEED = 42
NUM_TRAIN_SAMPLES = 2000
NUM_EVAL_SAMPLES = 100
MAX_SEQ_LENGTH = 384

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
dataset = load_dataset(DATASET_NAME, token=HF_TOKEN)
shuffled = dataset["train"].shuffle(seed=SEED)

train_data = shuffled.select(range(0, 2000))
eval_data = shuffled.select(range(2000, 2100))

assert len(train_data) == NUM_TRAIN_SAMPLES
assert len(eval_data) == NUM_EVAL_SAMPLES

print("Training samples:", len(train_data))
print("Validation samples:", len(eval_data))


In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def build_training_text(example):
    messages = [
        {
            "role": "system",
            "content": "You are a medical text simplification assistant."
        },
        {
            "role": "user",
            "content": (
                "Convert this radiology report into a patient-friendly "
                "layman summary.\n\nRadiology Report:\n"
                + str(example["radiology_report"]).strip()
            )
        },
        {
            "role": "assistant",
            "content": str(example["layman_report"]).strip()
        }
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_data = train_data.map(build_training_text)
eval_data = eval_data.map(build_training_text)


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto"
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


In [ ]:
warmup_steps = int(0.03 * (len(train_data) * 3 // (1 * 4)))

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=0.3,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer
)

train_result = trainer.train()
trainer.save_state()
trainer.model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR))


In [ ]:
del trainer
del model
del base_model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

inference_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=inference_bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

generation_model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    is_trainable=False
)
generation_model.eval()
generation_model.config.use_cache = True


In [ ]:
test_dataset = dataset["train"].shuffle(seed=42).select(range(10100, 10200))

reports = [str(value or "") for value in test_dataset["radiology_report"]]
references = [str(value or "") for value in test_dataset["layman_report"]]

assert len(reports) == 100
assert len(references) == 100
assert all(report.strip() for report in reports)
assert all(reference.strip() for reference in references)


In [ ]:
PREDICTION_PROMPT = """You are a medical communication assistant.

Convert the following radiology report into a clear, simple, patient-friendly explanation.

Rules:
1. Preserve all medical findings stated in the report.
2. Do not add any information that is not stated in the report.
3. Do not add symptoms, diagnoses, causes, treatments, recommendations, or prognosis.
4. Preserve negation, uncertainty, severity, location, and laterality.
5. Explain medical terms in simple language when needed.
6. Do not remove important findings.
7. Do not introduce information from outside the given report.
8. Return only the patient-friendly explanation.

Radiology Report:
{report}

Patient-Friendly Explanation:
"""

stop_phrases = [
    "\nRules:", "\n1. Preserve", "\n2. Do not", "\n3. Do not",
    "\n4. Preserve", "\n5. Explain", "\n6. Do not",
    "\n7. Do not", "\n8. Return", "\nRadiology Report:"
]

def generate_summary(report):
    prompt = PREDICTION_PROMPT.format(report=report)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=384
    )
    input_device = generation_model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}
    with torch.inference_mode():
        output = generation_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    generated_text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
    for phrase in stop_phrases:
        if phrase in generated_text:
            generated_text = generated_text.split(phrase)[0].strip()
    return generated_text

predictions = [
    generate_summary(report)
    for report in tqdm(reports, total=len(reports), desc="Generating 2K predictions")
]

assert len(predictions) == 100
assert all(prediction.strip() for prediction in predictions)

predictions_df = pd.DataFrame({
    "radiology_report": reports,
    "reference_layman_summary": references,
    "generated_summary": predictions
})

PREDICTION_PATH = RESULTS_DIR / "2k_finetuned_predictions_shared100_regenerated.csv"
predictions_df.to_csv(PREDICTION_PATH, index=False, encoding="utf-8-sig")


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

bleu = sacrebleu.corpus_bleu(predictions, [references]).score / 100.0

rouge_scorer_obj = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)
rouge_results = [
    rouge_scorer_obj.score(reference, prediction)
    for reference, prediction in zip(references, predictions)
]
rouge1 = float(np.mean([item["rouge1"].fmeasure for item in rouge_results]))
rouge2 = float(np.mean([item["rouge2"].fmeasure for item in rouge_results]))
rougeL = float(np.mean([item["rougeL"].fmeasure for item in rouge_results]))

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

meteor_scores = [
    meteor_score(
        [wordpunct_tokenize(reference.lower())],
        wordpunct_tokenize(prediction.lower())
    )
    for reference, prediction in zip(references, predictions)
]
meteor = float(np.mean(meteor_scores))

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    lang="en",
    model_type="roberta-large",
    device=DEVICE,
    batch_size=1,
    verbose=True
)
bert_precision = float(P.mean().item())
bert_recall = float(R.mean().item())
bert_f1 = float(F1.mean().item())
del P, R, F1
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

semantic_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=DEVICE
)
reference_embeddings = semantic_model.encode(
    references,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=16,
    show_progress_bar=True
)
prediction_embeddings = semantic_model.encode(
    predictions,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=16,
    show_progress_bar=True
)
semantic_scores = [
    float(util.cos_sim(prediction_embeddings[index], reference_embeddings[index]).item())
    for index in range(len(references))
]
semantic_similarity = float(np.mean(semantic_scores))

del semantic_model, reference_embeddings, prediction_embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

fkgl_scores = [textstat.flesch_kincaid_grade(value) for value in predictions]
dale_chall_scores = [textstat.dale_chall_readability_score(value) for value in predictions]
coleman_liau_scores = [textstat.coleman_liau_index(value) for value in predictions]

fkgl = float(np.mean(fkgl_scores))
dale_chall = float(np.mean(dale_chall_scores))
coleman_liau = float(np.mean(coleman_liau_scores))


In [ ]:
NLI_MODEL_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(DEVICE).eval()

id2label = {
    int(index): str(label).lower()
    for index, label in nli_model.config.id2label.items()
}
if not any("entail" in label for label in id2label.values()):
    raise ValueError(f"The NLI model does not expose expected entailment labels: {id2label}")

nli_rows = []
for report, prediction in tqdm(
    zip(reports, predictions),
    total=len(reports),
    desc="NLI evaluation"
):
    inputs = nli_tokenizer(
        report,
        prediction,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        probabilities = torch.softmax(nli_model(**inputs).logits[0], dim=-1).cpu().tolist()
    scores = {
        id2label[index]: float(probabilities[index])
        for index in range(len(probabilities))
    }
    predicted_label = max(scores, key=scores.get)
    nli_rows.append({
        "nli_label": predicted_label,
        "entailment_score": next((value for label, value in scores.items() if "entail" in label), 0.0),
        "neutral_score": next((value for label, value in scores.items() if "neutral" in label), 0.0),
        "contradiction_score": next((value for label, value in scores.items() if "contrad" in label), 0.0),
        "nli_non_entailment_flag": "entail" not in predicted_label
    })

nli_df = pd.DataFrame(nli_rows)
evaluated_df = pd.concat(
    [predictions_df.reset_index(drop=True), nli_df],
    axis=1
)
nli_non_entailment_count = int(evaluated_df["nli_non_entailment_flag"].sum())
nli_non_entailment_rate = 100.0 * nli_non_entailment_count / len(evaluated_df)

del nli_model, nli_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
metrics = {
    "Evaluation Samples": len(evaluated_df),
    "BLEU": bleu,
    "ROUGE-1": rouge1,
    "ROUGE-2": rouge2,
    "ROUGE-L": rougeL,
    "METEOR": meteor,
    "BERTScore Precision": bert_precision,
    "BERTScore Recall": bert_recall,
    "BERTScore F1": bert_f1,
    "Semantic Similarity (all-mpnet-base-v2)": semantic_similarity,
    "FKGL": fkgl,
    "Dale-Chall": dale_chall,
    "Coleman-Liau": coleman_liau,
    "NLI Non-Entailment Rate (%)": nli_non_entailment_rate
}

metrics_df = pd.DataFrame(
    [{"Metric": key, "Score": value} for key, value in metrics.items()]
)

METRICS_CSV = RESULTS_DIR / "2k_finetuned_metrics_shared100_regenerated.csv"
METRICS_JSON = RESULTS_DIR / "2k_finetuned_metrics_shared100_regenerated.json"
PREDICTIONS_WITH_NLI = RESULTS_DIR / "2k_finetuned_predictions_shared100_with_nli.csv"

metrics_df.to_csv(METRICS_CSV, index=False, encoding="utf-8-sig")
evaluated_df.to_csv(PREDICTIONS_WITH_NLI, index=False, encoding="utf-8-sig")

with open(METRICS_JSON, "w", encoding="utf-8") as file:
    json.dump(
        {
            "experiment": "2K fine-tuned Llama 3.2 1B",
            "evaluation_samples": len(evaluated_df),
            "dataset": DATASET_NAME,
            "shuffle_seed": 42,
            "test_selection": "dataset['train'].shuffle(seed=42).select(range(10100, 10200))",
            "adapter_path": str(ADAPTER_DIR),
            "bertscore_model": "roberta-large",
            "semantic_similarity_model": "sentence-transformers/all-mpnet-base-v2",
            "nli_model": NLI_MODEL_NAME,
            "nli_non_entailment_sample_count": nli_non_entailment_count,
            "nli_non_entailment_rate_percent": nli_non_entailment_rate,
            "nli_interpretation": "Model-based non-entailment proxy; not clinically validated hallucination detection.",
            "metrics": metrics
        },
        file,
        indent=2,
        ensure_ascii=False
    )

display(metrics_df)

print("Predictions:", PREDICTION_PATH)
print("Per-sample NLI predictions:", PREDICTIONS_WITH_NLI)
print("Metrics CSV:", METRICS_CSV)
print("Metrics JSON:", METRICS_JSON)

del generation_model
del base_model
gc.collect()
torch.cuda.empty_cache()
